# Notebook 02 — Chunking Experiment
Compare different chunking strategies on retrieval quality.
Tests chunk sizes: 256, 384 (default 512) tokens using a fixed set of queries.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from src.chunking.chunker import chunk_text, count_tokens

# Sample text from a Grab filing (replace with actual text from a parsed PDF)
SAMPLE_TEXT = """
Grab Holdings Limited reported strong financial results for the fiscal year ended December 31, 2023.
Total revenues for FY2023 were USD 2,357.8 million, representing a 65% increase compared to
FY2022 revenues of USD 1,430.3 million. The Group achieved its first full-year adjusted EBITDA
positivity, recording adjusted EBITDA of USD 33.2 million compared to a loss of USD 793.0 million
in FY2022. This represents a significant milestone in the company's path to sustainable profitability.
The Deliveries segment remained the largest revenue contributor with GMV of USD 8.9 billion.
Monthly Transacting Users reached 35.2 million in Q4 2023, reflecting continued ecosystem growth.
The company operates across eight countries in Southeast Asia including Singapore, Indonesia,
Malaysia, Thailand, Vietnam, Philippines, Cambodia, and Myanmar.
""" * 15  # Repeat to create enough text for chunking

EXPERIMENTS = [
    {'name': '256-token', 'strategy': 'fixed_token', 'chunk_size': 256, 'chunk_overlap': 30,
     'tokenizer': 'cl100k_base', 'min_chunk_tokens': 30, 'respect_sentence_boundaries': True},
    {'name': '512-token (default)', 'strategy': 'fixed_token', 'chunk_size': 512, 'chunk_overlap': 50,
     'tokenizer': 'cl100k_base', 'min_chunk_tokens': 50, 'respect_sentence_boundaries': True},
    {'name': '768-token', 'strategy': 'fixed_token', 'chunk_size': 768, 'chunk_overlap': 80,
     'tokenizer': 'cl100k_base', 'min_chunk_tokens': 80, 'respect_sentence_boundaries': True},
    {'name': 'sentence-window-5', 'strategy': 'sentence_window', 'window_size': 5,
     'overlap_sentences': 1, 'min_chunk_tokens': 30, 'tokenizer': 'cl100k_base'},
]

import statistics
print(f'{'Experiment':<25} {'N Chunks':>10} {'Avg Tokens':>12} {'Min':>8} {'Max':>8}')
print('-' * 65)
for exp in EXPERIMENTS:
    name = exp.pop('name')
    chunks = chunk_text(SAMPLE_TEXT, exp)
    token_counts = [c['token_count'] for c in chunks]
    exp['name'] = name  # Restore
    if token_counts:
        print(f'{name:<25} {len(chunks):>10} {statistics.mean(token_counts):>12.1f} {min(token_counts):>8} {max(token_counts):>8}')
    else:
        print(f'{name:<25} {0:>10}')

In [ ]:
# Visual comparison of chunk size distributions
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

EXPERIMENTS_PLOT = [
    {'label': '256-token', 'config': {'strategy': 'fixed_token', 'chunk_size': 256, 'chunk_overlap': 30, 'tokenizer': 'cl100k_base', 'min_chunk_tokens': 30, 'respect_sentence_boundaries': True}},
    {'label': '512-token (default)', 'config': {'strategy': 'fixed_token', 'chunk_size': 512, 'chunk_overlap': 50, 'tokenizer': 'cl100k_base', 'min_chunk_tokens': 50, 'respect_sentence_boundaries': True}},
    {'label': '768-token', 'config': {'strategy': 'fixed_token', 'chunk_size': 768, 'chunk_overlap': 80, 'tokenizer': 'cl100k_base', 'min_chunk_tokens': 80, 'respect_sentence_boundaries': True}},
    {'label': 'sentence-window-5', 'config': {'strategy': 'sentence_window', 'window_size': 5, 'overlap_sentences': 1, 'min_chunk_tokens': 30, 'tokenizer': 'cl100k_base'}},
]

for ax, exp in zip(axes, EXPERIMENTS_PLOT):
    chunks = chunk_text(SAMPLE_TEXT, exp['config'])
    counts = [c['token_count'] for c in chunks]
    ax.hist(counts, bins=20, color='#2E86C1', edgecolor='white')
    ax.set_title(exp['label'])
    ax.set_xlabel('Token count')
    ax.axvline(statistics.mean(counts) if counts else 0, color='red', linestyle='--')

plt.suptitle('Token Count Distribution by Chunking Strategy', fontsize=14)
plt.tight_layout()
plt.show()

## Retrieval Comparison
Compare how different chunk sizes affect retrieval on 5 test queries.
This requires indexes to already be built. Run once with each chunking config.

In [ ]:
# Test queries for retrieval comparison
TEST_QUERIES = [
    'What was Grab total revenue for FY2023?',
    'What are the regulatory risks for Grab in Southeast Asia?',
    'How did adjusted EBITDA change from FY2022 to FY2023?',
    'What is Grab monthly transacting user count?',
    'What guidance did Grab give for long-term GMV growth?',
]

# For each query, show top-3 chunk IDs and scores from the default 512-token index
try:
    from src.retrieval.dense_retriever import DenseRetriever
    retriever = DenseRetriever()
    for q in TEST_QUERIES:
        print(f'Q: {q}')
        chunks = retriever.retrieve(q, top_k=3)
        for i, c in enumerate(chunks, 1):
            m = c['metadata']
            print(f'  [{i}] score={c["score"]:.4f} | {m["doc_type"]} | {m["fiscal_period"]} | p{m["page_number"]} | {m["chunk_id"]}')
        print()
except Exception as e:
    print(f'Retrieval unavailable: {e}')
    print('Build the index first: python scripts/build_index.py')